# Bab 2 — Simulasi terbuka gelombang manusia

**ID unit:** `O005-LEGA-V101-CH02`  
**ID notebook:** `O005-LEGA-V101-CH02-NB01`  
**Rekaman sumber:** Pressbooks 27, *First Steps: Modeling the Wave*, dimodifikasi `2026-03-27T02:10:41Z`  
**Benih acak tetap:** `20260821`

Bab sumber menguji model gelombang manusia dengan `The_Wave.m` dan sebuah antarmuka MATLAB. Notebook ini menggantikan ketergantungan proprieter tersebut dengan implementasi Python terbuka, deterministik, dan ditulis secara independen. Ia bukan port atau terjemahan kode MATLAB. Seluruh perhitungan memakai NumPy dan Matplotlib serta dapat dijalankan tanpa jaringan setelah dependensi pada `requirements.lock` tersedia.

Teks dan matematika sumber berasal dari Joceline Lega, *Introduction to Mathematical Modeling*, versi 1.01, CC BY-NC-SA 4.0. Notebook baru ini didistribusikan bersama adaptasi Bahasa Indonesia dengan lisensi yang sama. Perubahan mencakup implementasi numerik, pilihan parameter pedagogis, grafik statis, dan pemeriksaan otomatis. Hasil ini tidak menyiratkan dukungan atau pengesahan oleh penulis maupun University of Arizona.

## Model yang dipertahankan

Penonton $i$ sedang ikut membentuk gelombang bila tingkat antusiasmenya melampaui ambang aktivitas $\mathcal A$. Setelah terpicu pada waktu $t_{\mathcal I}$, pusat denyutnya berada pada $t_0=t_{\mathcal I}+\Delta\tau$ dan

$$f(t)=\frac{1}{\cosh(b(t-t_0))}, \qquad x(i,t)=\begin{cases}f(t),&f(t)>\mathcal A,\\0,&f(t)\le\mathcal A.\end{cases}$$

Ambang tiap penonton ditarik sekali dari sebaran seragam $[c-\delta,c+\delta]$. Seorang penonton yang sedang duduk terpicu bila

$$w(i)=\sum_{\substack{j,\ |i-j|\le R_m\\x(j)>\mathcal A}}e^{-|i-j|/R}\left(1-\tanh(a(j-i))\right)>s_{th}(i).$$

Indeks kursi pada kode berjalan searah jarum jam; indeks kode 0 bersesuaian dengan kursi 1 pada grafik. Batas periodik diterapkan dengan operasi gulir melingkar. Untuk $a>0$, tetangga dengan offset $j-i<0$ lebih berpengaruh sehingga gelombang bergerak menuju indeks yang lebih besar. Semua kursi diperbarui serentak dari keadaan lama agar urutan iterasi tidak menciptakan arah semu.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

SEED = 20260821
np.set_printoptions(precision=5, suppress=True)

PARAMETER_DASAR = {
    "jumlah_kursi": 160,
    "durasi": 5.0,
    "langkah_waktu": 0.05,
    "ambang_rata_rata": 1.50,  # c
    "rentang_ambang": 0.04,    # delta
    "ambang_aktivitas": 0.35,  # A
    "b": 4.0,
    "waktu_reaksi": 0.35,      # Delta tau
    "R": 2.0,
    "R_m": 3,
    "a": 2.0,
    "kursi_awal": 12,          # indeks kode berbasis nol
    "jumlah_awal": 3,
}

print(f"Benih acak: {SEED}")
print(f"NumPy {np.__version__}")
print(f"Perkiraan durasi sumber, tau ~ 10/b: {10 / PARAMETER_DASAR['b']:.2f} s")

In [ ]:
def sech(z):
    """Secant hiperbolik yang stabil untuk argumen numerik besar."""
    return 1.0 / np.cosh(np.clip(z, -700.0, 700.0))


def bobot_kernel(offset, R, a):
    """Bobot pengaruh untuk offset periodik bertanda j-i."""
    offset = np.asarray(offset, dtype=float)
    return np.exp(-np.abs(offset) / R) * (1.0 - np.tanh(a * offset))


def pengaruh_tetangga(x, ambang_aktivitas, R, R_m, a):
    """Hitung w(i) persis dari tetangga aktif pada cincin kursi."""
    aktif = (x > ambang_aktivitas).astype(float)
    w = np.zeros_like(x, dtype=float)
    for offset in range(-R_m, R_m + 1):
        # np.roll(..., -offset)[i] mengambil keadaan j=i+offset.
        w += bobot_kernel(offset, R, a) * np.roll(aktif, -offset)
    return w

In [ ]:
def simulasikan(parameter, seed=SEED):
    """Simulasi sinkron model gelombang manusia dengan batas periodik."""
    p = dict(parameter)
    N = int(p["jumlah_kursi"])
    dt = float(p["langkah_waktu"])
    durasi = float(p["durasi"])
    R_m = int(p["R_m"])

    if N <= 2 * R_m + 1:
        raise ValueError("jumlah_kursi harus lebih besar daripada 2*R_m+1")
    if not (dt > 0.0 and durasi > 0.0 and p["b"] > 0.0 and p["R"] > 0.0):
        raise ValueError("dt, durasi, b, dan R harus positif")
    if not (0.0 < p["ambang_aktivitas"] < 1.0):
        raise ValueError("ambang aktivitas harus berada di antara 0 dan 1")
    if not (0 < p["jumlah_awal"] < N):
        raise ValueError("jumlah penonton awal harus berada di antara 1 dan N-1")

    jumlah_langkah = int(round(durasi / dt))
    if not np.isclose(jumlah_langkah * dt, durasi):
        raise ValueError("durasi harus merupakan kelipatan langkah_waktu")
    waktu = np.linspace(0.0, durasi, jumlah_langkah + 1)

    rng = np.random.default_rng(seed)
    ambang = rng.uniform(
        p["ambang_rata_rata"] - p["rentang_ambang"],
        p["ambang_rata_rata"] + p["rentang_ambang"],
        N,
    )

    waktu_puncak = np.full(N, np.nan)
    dalam_siklus = np.zeros(N, dtype=bool)
    waktu_picuan_pertama = np.full(N, np.nan)
    kursi_awal = (p["kursi_awal"] + np.arange(p["jumlah_awal"])) % N
    waktu_puncak[kursi_awal] = 0.0
    dalam_siklus[kursi_awal] = True
    waktu_picuan_pertama[kursi_awal] = 0.0

    aktivitas = np.zeros((jumlah_langkah + 1, N))
    pengaruh = np.zeros_like(aktivitas)

    for nomor, t in enumerate(waktu):
        x = np.zeros(N)
        peserta = np.flatnonzero(dalam_siklus)
        if peserta.size:
            fase = t - waktu_puncak[peserta]
            nilai_denyut = sech(p["b"] * fase)
            tampak = nilai_denyut > p["ambang_aktivitas"]
            x[peserta[tampak]] = nilai_denyut[tampak]

            # Setelah melewati puncak dan turun di bawah A, penonton duduk lagi.
            selesai = (fase > 0.0) & (~tampak)
            if np.any(selesai):
                dalam_siklus[peserta[selesai]] = False
                waktu_puncak[peserta[selesai]] = np.nan

        w = pengaruh_tetangga(
            x, p["ambang_aktivitas"], p["R"], R_m, p["a"]
        )
        aktivitas[nomor] = x
        pengaruh[nomor] = w

        # Pemicu dihitung serentak dari keadaan saat ini dan berlaku sesudahnya.
        baru = np.flatnonzero((~dalam_siklus) & (w > ambang))
        if baru.size:
            dalam_siklus[baru] = True
            waktu_puncak[baru] = t + p["waktu_reaksi"]
            belum_tercatat = np.isnan(waktu_picuan_pertama[baru])
            waktu_picuan_pertama[baru[belum_tercatat]] = t

    return {
        "waktu": waktu,
        "ambang": ambang,
        "aktivitas": aktivitas,
        "pengaruh": pengaruh,
        "waktu_picuan_pertama": waktu_picuan_pertama,
        "kursi_awal": kursi_awal,
    }

## Simulasi dasar dan pengukuran

Nilai dasar di atas dipilih untuk demonstrasi yang dapat diperiksa, bukan sebagai klaim bahwa nilai tersebut identik dengan bawaan perangkat sumber. Tiga penonton mulai pada puncak denyut. Kita ukur lebar sebagai median jumlah penonton aktif setelah transien awal, dan kecepatan dari regresi jarak terhadap waktu picuan pertama. Rentang regresi dihentikan sebelum setengah lingkaran sehingga batas periodik tidak mengaburkan arah.

In [ ]:
def ukur_kecepatan(hasil, parameter, arah):
    """Ukur besar kecepatan sepanjang arah +1 atau -1 dalam kursi/detik."""
    N = int(parameter["jumlah_kursi"])
    kursi = np.arange(N)
    if arah == 1:
        asal = (parameter["kursi_awal"] + parameter["jumlah_awal"] - 1) % N
    elif arah == -1:
        asal = parameter["kursi_awal"] % N
    else:
        raise ValueError("arah harus +1 atau -1")

    jarak = (arah * (kursi - asal)) % N
    waktu_picuan = hasil["waktu_picuan_pertama"]
    batas = min(70, N // 2 - 1)
    dipakai = np.isfinite(waktu_picuan) & (jarak >= 5) & (jarak <= batas)
    if np.count_nonzero(dipakai) < 4:
        raise ValueError("terlalu sedikit kursi terpicu untuk mengukur kecepatan")

    kemiringan, intersep = np.polyfit(waktu_picuan[dipakai], jarak[dipakai], 1)
    taksiran = kemiringan * waktu_picuan[dipakai] + intersep
    sse = np.sum((jarak[dipakai] - taksiran) ** 2)
    sst = np.sum((jarak[dipakai] - np.mean(jarak[dipakai])) ** 2)
    return float(kemiringan), float(1.0 - sse / sst)


hasil_dasar = simulasikan(PARAMETER_DASAR)
aktif_dasar = hasil_dasar["aktivitas"] > PARAMETER_DASAR["ambang_aktivitas"]
jumlah_aktif = np.sum(aktif_dasar, axis=1)
jendela_mapan = (hasil_dasar["waktu"] >= 1.0) & (hasil_dasar["waktu"] <= 4.5)
lebar_median = float(np.median(jumlah_aktif[jendela_mapan]))
kecepatan, r2_kecepatan = ukur_kecepatan(hasil_dasar, PARAMETER_DASAR, arah=1)
jumlah_terpicu = int(np.count_nonzero(np.isfinite(hasil_dasar["waktu_picuan_pertama"])))

print(f"Penonton yang pernah terpicu : {jumlah_terpicu}")
print(f"Lebar median gelombang      : {lebar_median:.1f} kursi")
print(f"Kecepatan terukur           : {kecepatan:.1f} kursi/detik")
print(f"R² regresi kecepatan        : {r2_kecepatan:.6f}")

In [ ]:
indeks_cuplikan = int(np.argmin(np.abs(hasil_dasar["waktu"] - 2.5)))
waktu_cuplikan = hasil_dasar["waktu"][indeks_cuplikan]
kursi = np.arange(1, PARAMETER_DASAR["jumlah_kursi"] + 1)
waktu_relatif = np.linspace(-1.5, 1.5, 500)
offset_rapat = np.linspace(-3.5, 3.5, 500)
offset_diskret = np.arange(-PARAMETER_DASAR["R_m"], PARAMETER_DASAR["R_m"] + 1)

fig_diagnostik, axes = plt.subplots(2, 2, figsize=(13, 8), constrained_layout=True)

axes[0, 0].bar(kursi, hasil_dasar["aktivitas"][indeks_cuplikan], color="#0072B2")
axes[0, 0].axhline(PARAMETER_DASAR["ambang_aktivitas"], color="#D55E00",
                   linestyle="--", label=r"ambang $\mathcal{A}$")
axes[0, 0].set(title=f"Aktivitas pada t = {waktu_cuplikan:.2f} s",
               xlabel="nomor kursi", ylabel=r"$x(i,t)$")
axes[0, 0].legend()

axes[1, 0].plot(kursi, hasil_dasar["ambang"], color="#009E73", linewidth=1.5)
axes[1, 0].axhline(PARAMETER_DASAR["ambang_rata_rata"], color="#222222",
                   linestyle="--", label="c")
axes[1, 0].set(title="Ambang antusiasme tetap", xlabel="nomor kursi", ylabel=r"$s_{th}(i)$")
axes[1, 0].legend()

denyut = sech(PARAMETER_DASAR["b"] * waktu_relatif)
axes[0, 1].plot(waktu_relatif, denyut, color="#CC79A7", linewidth=2.5)
axes[0, 1].axhline(PARAMETER_DASAR["ambang_aktivitas"], color="#D55E00", linestyle="--")
axes[0, 1].set(title="Denyut satu penonton", xlabel=r"$t-t_0$ (s)", ylabel=r"$f(t)$")

kernel_rapat = bobot_kernel(offset_rapat, PARAMETER_DASAR["R"], PARAMETER_DASAR["a"])
kernel_diskret = bobot_kernel(offset_diskret, PARAMETER_DASAR["R"], PARAMETER_DASAR["a"])
axes[1, 1].plot(offset_rapat, kernel_rapat, color="#222222", linewidth=2)
axes[1, 1].scatter(offset_diskret, kernel_diskret, color="#D55E00", s=55, zorder=3,
                   label=r"offset $|j-i|\leq R_m$")
axes[1, 1].set(title="Kernel pengaruh asimetris", xlabel=r"offset $j-i$", ylabel="bobot")
axes[1, 1].legend()

for ax in axes.flat:
    ax.grid(alpha=0.25)
fig_diagnostik.suptitle("Diagnostik terbuka pengganti panel GUI sumber", fontsize=15)
plt.show()

**Deskripsi panjang gambar diagnostik:** Empat panel statis menggantikan fungsi informasional GUI sumber. Panel kiri atas memperlihatkan satu kelompok sekitar 15 batang aktivitas biru yang bergerak di sepanjang 160 kursi pada $t=2{,}50$ detik; garis jingga putus-putus menandai ambang aktivitas. Panel kiri bawah memperlihatkan ambang antusiasme tiap kursi yang berfluktuasi kecil dan deterministik di sekitar garis rata-rata $c=1{,}50$. Panel kanan atas memperlihatkan denyut $1/\cosh(4(t-t_0))$ yang simetris, berpuncak satu pada $t=t_0$, lalu memotong ambang aktivitas pada kedua sisinya. Panel kanan bawah memperlihatkan kernel pengaruh: bobot pada offset negatif jauh lebih besar daripada pada offset positif, sehingga arah rambatan tidak simetris.

## Eksperimen parameter terbatas

Empat panel berikut menguji pertanyaan inti bab. Kasus dasar harus merambat searah jarum jam. Mengubah tanda $a$ harus membalik arah. Menaikkan $c$ cukup jauh atau memperpendek jangkauan $R$ harus mematikan gelombang karena pengaruh tetangga tidak lagi melampaui ambang. Semua kasus memakai ambang acak yang sama karena benihnya sama.

In [ ]:
parameter_balik = {**PARAMETER_DASAR, "a": -PARAMETER_DASAR["a"]}
parameter_ambang_tinggi = {**PARAMETER_DASAR, "ambang_rata_rata": 2.60}
parameter_jangkauan_pendek = {**PARAMETER_DASAR, "R": 1.0}

hasil_balik = simulasikan(parameter_balik)
hasil_ambang_tinggi = simulasikan(parameter_ambang_tinggi)
hasil_jangkauan_pendek = simulasikan(parameter_jangkauan_pendek)

kasus = [
    ("Dasar: a = +2", hasil_dasar, PARAMETER_DASAR),
    ("Arah dibalik: a = -2", hasil_balik, parameter_balik),
    ("Ambang tinggi: c = 2,60", hasil_ambang_tinggi, parameter_ambang_tinggi),
    ("Jangkauan pendek: R = 1", hasil_jangkauan_pendek, parameter_jangkauan_pendek),
]

fig_ruang_waktu, axes = plt.subplots(2, 2, figsize=(13, 9), sharex=True, sharey=True,
                                      constrained_layout=True)
for ax, (judul, hasil, parameter) in zip(axes.flat, kasus):
    gambar = ax.imshow(
        hasil["aktivitas"],
        origin="lower",
        aspect="auto",
        extent=[1, parameter["jumlah_kursi"], 0, parameter["durasi"]],
        vmin=0.0,
        vmax=1.0,
        cmap="viridis",
    )
    pernah = np.count_nonzero(np.isfinite(hasil["waktu_picuan_pertama"]))
    ax.set_title(f"{judul} — {pernah} kursi terpicu")
    ax.set_xlabel("nomor kursi")
    ax.set_ylabel("waktu (s)")

fig_ruang_waktu.colorbar(gambar, ax=axes, label=r"tingkat antusiasme $x(i,t)$", shrink=0.88)
fig_ruang_waktu.suptitle("Diagram ruang–waktu untuk empat pilihan parameter", fontsize=15)
plt.show()

**Deskripsi panjang diagram ruang–waktu:** Empat peta panas memakai nomor kursi pada sumbu mendatar, waktu pada sumbu tegak, dan warna gelap-ke-kuning untuk tingkat antusiasme nol-ke-satu. Kasus dasar membentuk pita diagonal naik ke kanan, yang berarti gelombang merambat menuju nomor kursi lebih besar. Ketika tanda $a$ dibalik, pita bergerak ke kiri, melintasi batas periodik dekat $t=1{,}2$ detik, lalu muncul kembali dari sisi kanan dan terus bergerak ke kiri. Pada kasus ambang tinggi dan jangkauan pendek, hanya pita pendek dari tiga kursi awal yang tampak dekat waktu nol; tidak terbentuk gelombang merambat.

## Pemeriksaan yang dapat dieksekusi

Pemeriksaan berikut mencakup determinisme, domain nilai, sebaran ambang, batas periodik, orientasi kernel, rambatan dan lebar kasus dasar, pembalikan arah, serta dua kasus pemadaman. Toleransi kecepatan mengizinkan variasi numerik kecil antaraplikasi, tetapi akan menangkap perubahan aturan waktu.

In [ ]:
# Determinisme dan bentuk keluaran.
hasil_ulang = simulasikan(PARAMETER_DASAR, seed=SEED)
assert np.array_equal(hasil_dasar["ambang"], hasil_ulang["ambang"])
assert np.array_equal(hasil_dasar["aktivitas"], hasil_ulang["aktivitas"])
assert np.array_equal(
    hasil_dasar["waktu_picuan_pertama"],
    hasil_ulang["waktu_picuan_pertama"],
    equal_nan=True,
)
assert hasil_dasar["aktivitas"].shape == (101, 160)

# Nilai model dan distribusi ambang.
assert np.all(np.isfinite(hasil_dasar["aktivitas"]))
assert np.min(hasil_dasar["aktivitas"]) >= 0.0
assert np.max(hasil_dasar["aktivitas"]) <= 1.0
assert np.all(
    (hasil_dasar["aktivitas"] == 0.0)
    | (hasil_dasar["aktivitas"] > PARAMETER_DASAR["ambang_aktivitas"])
)
assert np.min(hasil_dasar["ambang"]) >= 1.46
assert np.max(hasil_dasar["ambang"]) <= 1.54

# Kernel dan batas periodik: kursi terakhir adalah tetangga offset -1 kursi pertama.
assert bobot_kernel(-1, R=2.0, a=2.0) > bobot_kernel(1, R=2.0, a=2.0)
uji_cincin = np.zeros(8)
uji_cincin[-1] = 1.0
w_uji = pengaruh_tetangga(uji_cincin, 0.35, R=2.0, R_m=3, a=2.0)
assert np.isclose(w_uji[0], bobot_kernel(-1, R=2.0, a=2.0))

# Kasus dasar merambat dengan lebar yang sebanding dengan sekitar 15 kursi sumber.
assert jumlah_terpicu >= 100
assert 11.0 <= lebar_median <= 22.0
assert 19.5 <= kecepatan <= 20.5
assert r2_kecepatan > 0.999

# Membalik a membalik arah tanpa mengubah besar kecepatan pada parameter simetris lainnya.
kecepatan_balik, r2_balik = ukur_kecepatan(hasil_balik, parameter_balik, arah=-1)
jumlah_terpicu_balik = np.count_nonzero(np.isfinite(hasil_balik["waktu_picuan_pertama"]))
assert jumlah_terpicu_balik >= 100
assert np.isclose(kecepatan_balik, kecepatan, rtol=0.01)
assert r2_balik > 0.999
assert np.isnan(hasil_dasar["waktu_picuan_pertama"][PARAMETER_DASAR["kursi_awal"] - 1])
assert np.isnan(hasil_balik["waktu_picuan_pertama"][(PARAMETER_DASAR["kursi_awal"] + PARAMETER_DASAR["jumlah_awal"]) % 160])

# Ambang tinggi dan jangkauan pendek tidak menyalakan kursi di luar pemicu awal.
assert np.count_nonzero(np.isfinite(hasil_ambang_tinggi["waktu_picuan_pertama"])) == 3
assert np.count_nonzero(np.isfinite(hasil_jangkauan_pendek["waktu_picuan_pertama"])) == 3

print("Semua pemeriksaan lulus: model deterministik, periodik, terarah, dan peka parameter.")

## Kesimpulan

Dengan pilihan dasar, gelombang selebar median 15 kursi merambat sekitar 20 kursi per detik—dekat dengan pengamatan sekitar 22 kursi per detik yang disebutkan bab, tetapi bukan hasil kalibrasi terhadap rekaman stadion. Langkah waktu, bentuk denyut, waktu reaksi, ambang, dan kernel bekerja sebagai kombinasi; mengubah satu parameter dapat mengubah kecepatan atau mematikan rambatan. Membalik tanda $a$ membalik arah karena sisi kernel yang kuat ikut berbalik.

Untuk eksplorasi lanjutan, salin `PARAMETER_DASAR`, ubah satu nilai saja, lalu panggil `simulasikan`. Secara khusus, bandingkan beberapa nilai `jumlah_kursi`, `R`, `R_m`, `b`, `waktu_reaksi`, `langkah_waktu`, `ambang_aktivitas`, dan `jumlah_awal`. Pisahkan pengaruh numerik langkah waktu dari skala fisik model; hasil simulasi bukan bukti bahwa satu set parameter merupakan satu-satunya penjelasan gelombang stadion.